# Lab 5 — Three detectors, one dataset
### EP0418 · Topic 5 · 2 practical hours

**You need:** nothing to download: the notebook fetches `ep0418.py` and
`smart_building_sensors.csv` itself. Have the printed Lab 5 sheet beside you.

---
Today you build three anomaly detectors on the same classroom sensor file, each one
a step smarter than the one before, and compare **what each one catches and what
each one misses** (Tasks 1 to 4). Then you build a fourth detector for a fault that
none of the three can see (Task 5). The comparison matters more than the code:
Gemini writes the code.

## How the cells work

- **Grey code cells:** run each one when you reach it, top to bottom. Click in it
  and press Shift+Enter.
- **Text boxes** have a bold label ending in a colon, like **My answers:**.
  Double-click, type, press Shift+Enter. Or write on the printed sheet.
- **Gemini's code** arrives as a **green cell**. Click **Accept and run** (bottom of
  the chat panel), then run the task's check cell.
- **Check cells** start with `# TODO(you):`, which says what to run first. Never
  change them: if Gemini shows red and green lines inside one, click **Cancel**
  (red and green lines inside Gemini's own green cell are fine). Leave lines like
  `s = keep("s")` as they are. If a check cell says *Not there yet* or *No ... yet*,
  type in the chat *call the result* and the name the task asks for, accept, and
  run the check cell again.
- **Gemini's lab rules** (*EP0418 lab rules*) are built in. If a box *Review custom
  instructions* appears, click **Use custom instruction**. The sliders icon beside
  **+** in the chat box should be blue; if it is grey, click it and choose *EP0418
  lab rules*. Keep *Learn mode* off.
- **If Gemini says *Daily limit reached*:** paste the same prompt into
  ChatGPT, Claude or Copilot in a new tab. It cannot see your notebook: copy its code
  into a new cell (**+ Code** at the top of the page) and run it, and paste any
  failed check or red error back into its chat. Do the quiz last to save your limit.
- **Tables to fill in:** double-click the table. Each row looks like `| Task 1 |  |`.
  Type your answer between the last two `|`, so it becomes `| Task 1 | 13 gauges |`.
  Do not delete any `|`, or the `|---|` line. Press Shift+Enter to see the table.
- **Prompts** in grey quote boxes start with **Rules:** (for example, do not drop
  rows; show the row count before and after), then **Task:**. Keep the Rules when
  you change a prompt. A prompt with no example is **zero-shot**; one example of the
  output you want makes it **one-shot**, several make it **few-shot**.
- **Checks** (the `check.*` lines under a task) are the **verification contract**.
  Do not edit them. In industry they are called **evals**.
- **A failing check** says what it found and what to try. Failing checks are normal
  while you work.

> **First: File → Save a copy in Drive, and work in your copy.** This page opened from GitHub, and changes here are not saved. If a box *Review custom instructions* appears while you save the copy, click **Make a copy**: it keeps the lab's short rules for the AI assistant. Colab opens your copy in a new tab with *Copy of* at the start of its name at the top left: work in that tab and close the old one. Reopen your copy from Google Drive next time.

> **Then run the grey code cell just below** (click in it and press Shift+Enter, or click the play button at its left). Colab may warn that the notebook was not authored by Google: click **Run anyway**. You should see nothing in red; lines like *fetched ...*, a row count or a short table are normal. That means the course files and tools are loaded. Every lab starts this way. From here on, **run every grey code cell when you reach it**, top to bottom. **To write in a text box** (a cell with a bold label ending in a colon), double-click it, type, then press Shift+Enter.

In [ ]:
import os, urllib.request
REPO = "https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/"


def fetch(*names):
    """Download any course file that is not beside the notebook yet."""
    for name in names:
        if os.path.exists(name):
            continue
        try:
            urllib.request.urlretrieve(REPO + name, name)
            print("fetched", name)
        except Exception:
            print(f"Could not download {name}. Upload it with the folder icon on the left.")
            from google.colab import files
            files.upload()


fetch("ep0418.py")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from ep0418 import check, show


def keep(name):
    """The value Gemini's cell already made under this name, or ... if there is none yet."""
    global _NAMED
    v = globals().get(name, ...)
    if v is ... or v is None:
        print(f"Not there yet: nothing called {name}. Not run Gemini's green cell yet? Click Accept and "
              f"run on it, then run this cell again. Already ran it? Type in the same chat: call the "
              f"result {name}. Then Accept and run, and run this cell again.")
        _NAMED = True
        return ...
    return v


def filled(*values):
    """True once every TODO above has a value. Otherwise says what to do."""
    global _NAMED
    if any(v is ... or v is None for v in values):
        if not globals().get("_NAMED"):      # keep() has already said exactly what is missing
            print("Not there yet. Run Gemini's code first (click Accept and run on its green cell, "
                  "or paste its code into a new cell and run it), then run this cell again.")
        _NAMED = False
        return False
    _NAMED = False
    return True


def show_predict(data, unit):
    """Prediction League: run the newest predict that Gemini wrote and print its code to copy."""
    ran = globals().get("In", [])                                   # every cell you have run, oldest first
    mine = [c for c in ran if "def predict(" in c and "The baseline predicts" not in c and "show_predict" not in c]
    if not mine:
        print("Gemini's predict has not run yet. Find the cell Gemini added in Step 2 (it starts with")
        print("def predict, or with import lines just above it), click in it, press Shift+Enter,")
        print("then run this cell again."); return
    code = mine[-1].strip()
    for short, line in (("np.", "import numpy as np"), ("pd.", "import pandas as pd")):
        if short in code and line not in code:                     # the league page starts empty
            code = line + "\n" + code
    ns = {}
    exec(code, ns)
    print("Your predict says", round(ns["predict"](data.copy()), 1), unit)
    print()
    print("----- copy from the line below -----")
    print(code)
    print("----- to the line above -----")

CSV = "smart_building_sensors.csv"
fetch(CSV)

raw = pd.read_csv(CSV, parse_dates=["timestamp"])

# The Lab 4 cleaning is already done for you here: -999 becomes NaN, duplicates dropped.
df = raw.copy()
df.loc[df.humidity_pct == -999, "humidity_pct"] = np.nan
df = df.drop_duplicates(subset="timestamp", keep="first").sort_values("timestamp")
print(f"{len(df):,} rows ready")

# Placeholders so later cells give a readable message, not a NameError.
alert_fixed = alert_z = alert_ctx = z = stuck = None

**What each field means.** Read this before Task 1. The file is one classroom's
sensors, one row every 5 minutes, from Friday 1 May to Monday 29 June 2026: 17,214
rows once cleaned.

| Field | What it means | Example |
|---|---|---|
| `timestamp` | when the reading was taken, Singapore time, every 5 minutes | 2026-05-01 00:00 |
| `co2_ppm` | carbon dioxide in the room air, in parts per million. People breathe it out, so it rises when the room fills | about 420 at night, about 1,000 on a busy afternoon |
| `occupancy` | how many people are in the room; 0 means empty | 0 at night and at weekends, up to 45 |
| `temp_c` | air temperature, in °C | 24.9 |
| `humidity_pct` | how damp the air is, in %. In the file, -999 means no reading; the first cell has already made those blank | 76.3 |
| `energy_kwh` | electricity the room used in those 5 minutes, in kWh | 2.1 |

Tasks 1 to 4 use `co2_ppm` and `occupancy`. Task 5 uses `temp_c`.

## Task 1 — Build detector 1: a fixed threshold (15 minutes) · `PROMPT` `GENERATE` `VERIFY` `REFINE`

*Topic 5, section I: a fixed threshold.*

**Why a detector?** The file has 17,214 readings, and nobody can read them all. A
detector reads every one and marks the few that look wrong, so a person only checks
those. In a real building the detector judges each reading as it arrives; here it
runs once on the saved file, so everyone gets the same counts.

The simplest detector is one fixed line: alarm whenever CO2 is above 1000 ppm. You
want `alert_fixed`, True for every reading above 1000 and False for the rest. Tags show where you are in the loop: `PROMPT` → `GENERATE` → `VERIFY` → `REFINE`.

The prompts in this lab are **zero-shot**: they give no example of the output. That
is enough here, because each answer is one True or False, or one number, per row.

**Step 1.** `PROMPT` **Ask Gemini.** Open Gemini (the **spark icon** at the bottom centre of the Colab window), start a new chat (click **⋮** at the top right of the Gemini panel, then **New chat**, then **Yes**), paste this and press Enter. To copy it, drag
across the quoted text below from its first word to its last, then press Ctrl+C
(Cmd+C on a Mac):

> **Rules:** Do not drop any rows. Explain in plain words what the code does.
>
> **Task:** `df` is a DataFrame with a `co2_ppm` column. Make `alert_fixed`, a
> True/False Series that is True for every reading where co2_ppm is above 1000.

**Step 2.** `GENERATE` **Accept its code.** Gemini explains its plan in the chat and puts its code
into the notebook as a **green cell**. Click **Accept and run**: the button sits at
the bottom of the chat panel, just above the box where you type. It does not matter
where in the notebook the green cell lands. Ignore the blue suggestion chips under
its answer. If the code appears only in the chat, click the copy icon on the code,
click once on this text, click **+ Code** at the top of the page, paste into the new
cell, and run it. If Gemini ever shows red and green lines inside the check cell
(first line `# TODO(you)`), click **Cancel**: check cells are never changed.
Colab scrolls to the green cell, even to the very top of the notebook, so after
**Accept and run** scroll back down to this task for Step 3.

**Step 3.** `VERIFY` **Run the check cell below.** You should see the alert count, the count while the room was empty, alerts per week, and a chart with flags.

**Step 4.** `REFINE` **Only if a check fails, a number looks wrong, or Gemini's green cell
shows red error text:** in the same chat, type one
sentence saying what you saw and what you expected, for example *alert_fixed has 17,214 Trues; it should only be True where co2_ppm is above 1000.* Press Enter,
click **Accept and run** (Gemini may add a new green cell or change its own earlier
one; either is fine), and run the check cell below again. If everything
passed, there is nothing to fix: move on.

In [ ]:
# TODO(you): Accept and run Gemini's green cell first (wherever it landed) (it makes `alert_fixed`), then run this cell.
alert_fixed = keep("alert_fixed")
if filled(alert_fixed):
    print(f"alerts: {alert_fixed.sum():,} of {len(df):,}  ({alert_fixed.mean():.1%})")
    print(f"fired while room EMPTY: {(alert_fixed & (df.occupancy == 0)).sum()}")
    print()
    print("alerts per week (watch what happens over time):")
    print(alert_fixed.groupby(df.timestamp.dt.isocalendar().week).sum().to_string())
    show.flags(df, alert_fixed, col="co2_ppm", title="Detector 1: fixed threshold at 1000 ppm")

**What the cell above just did**, on two Tuesdays and eight weeks (it plays slowly, about a minute):
![A fixed line at 1000 ppm: 37 alerts on 12 May, 86 on 23 June after the sensor drifted up by about 100 ppm, and the weekly count climbing from 126 to 187 to 383 and 450](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab05_fixed_line.gif)

**Expected output** *(computed from the data file when this notebook was built)*:

- **Line 1:** alerts: 1,873 of 17,214 (10.9%).
- **Line 2:** fired while room EMPTY: 20.
- **Then one line per week:** 126 to 187 alerts in each full week of May and early June, then 383 and 450 in the last two full weeks. The first and last lines are part-weeks (the file starts on a Friday and ends on a Monday): leave them out.
- **The chart:** the CO2 line with a flag on every alert.

**What you should see.** Far more flags from 15 June on than before. The room
did not get busier: at night, with the room empty, CO2 read about 420 ppm on 12 May
but about 527 ppm on 23 June. The sensor drifted up by about 100 ppm, so more
ordinary afternoons now cross 1000. A fixed line cannot tell the difference, so the
weekly count climbs. Detectors 2 and 3 try to tell a drifting sensor from a busy room.

## Task 2 — Build detector 2: a rolling z-score (25 minutes) · `PROMPT` `GENERATE` `VERIFY` `REFINE`

*Topic 5, section II: a rolling z-score compares each reading with the recent past, looking back only.*

**Why another detector?** Detector 1 raised more alarms each week only because the
sensor drifted, not because the room changed. A better detector judges each reading
against what has been normal recently, not against one number fixed for two months.

Compare each reading against its own recent past instead of a fixed number. A
**z-score** says how unusual a reading is: how many "typical wobbles" (standard
deviations) it sits away from the recent average. Above 3, or below −3, is rare. You
want `z` and `alert_z`, with a window of the last 24 hours.

**Step 1. Predict, before you ask the AI.** Write two short answers in the box below:

1. The rows are 5 minutes apart. How many rows make 24 hours?
2. Detector 1's weekly count climbed from about 150 to 450 as the sensor drifted.
   Will a detector that compares each reading with its own last 24 hours climb the
   same way? Give your reason.

**My plan:**

*(Your answer box: double-click this cell. Type your answer here.)*

**Step 2.** `PROMPT` **Ask Gemini for the code.** Open Gemini (the **spark icon** at
the bottom centre of the Colab window) and start a new chat (click **⋮** at the top
right of the Gemini panel, then **New chat**, then **Yes**). The prompt below ends
with one sentence hidden: how many rows the 24-hour window holds, and whether it may
use readings that come after the one it judges. **Before you open it, say your
answer** (question 1 of your plan in Step 1 gives the first half).

<details><summary><b>Show the prompt</b> (click, then copy all of it into the chat and press Enter)</summary>

> **Rules:** Do not drop any rows. Say how many rows make 24 hours. Explain in plain
> words what the code does.
>
> **Task:** `df` has a `co2_ppm` column with one row every five minutes. Compute a
> rolling mean and rolling standard deviation of co2_ppm over a 24-hour window, then
> `z`, the z-score of each reading, and `alert_z`, True where the absolute value of z
> is above 3. The 24-hour window must be the last 288 rows, ending at each reading.

**The last sentence was the hidden one.** It says two things. Did your answer say both?

- **288 rows**: 12 rows an hour times 24 hours. `rolling(24)` counts rows, not
  hours, so 24 rows would be only 2 hours.
- **ending at each reading**: the window looks back only. A window centred on each
  reading uses readings that have not happened yet, so it could never run live. On
  this file it also fires only about 6 times.
</details>

**Step 3.** `GENERATE` **Run Gemini's code.**

1. Gemini explains its plan in the chat and adds a new code cell to your notebook,
   with a green edge. It does not matter where in the notebook it lands. Click
   **Accept and run**: the button sits at the bottom of the chat panel, just above
   the box where you type. This keeps the code and runs it. Ignore the blue
   suggestion chips under Gemini's answer.
2. If the code appears only in the chat and no green cell appears: click the copy
   icon on the code, click once on this text, click **+ Code** at the top of the
   page, paste into the new cell, and run it.
3. If Gemini shows red and green lines inside the check cell (first line
   `# TODO(you)`), click **Cancel**: check cells are never changed.
4. Colab scrolls to the green cell, even to the very top of the notebook. Scroll
   back down to Task 2 for Step 4.

**Step 4.** `VERIFY` **Run the check cell below.** You should see a few hundred z-score alerts, far fewer than the fixed threshold, and a chart with flags.

**Step 5.** `REFINE` **Only if your outcome is B or C in the table *Compare your outcome*, or Gemini's green cell
shows red error text:** in the same chat, type the fix sentence that table gives (for red
text, paste its last line with *fix this*). Press Enter, click **Accept and run**, and run
the check cell again. If you got A, there is nothing to fix: move on.

In [ ]:
# TODO(you): Accept and run Gemini's green cell first (wherever it landed) (it makes `z` and `alert_z`), then run this cell.
z = keep("z")
alert_z = keep("alert_z")

if filled(z, alert_z):
    print(f"z-score alerts: {alert_z.sum():,}")
    print(f"fixed alerts  : {alert_fixed.sum():,}")
    show.flags(df, alert_z, col="co2_ppm", title="Detector 2: rolling z-score")

**What the cell above just did**, on two real readings and eight weeks (it plays slowly, about a minute):
![A rolling z-score: 288 rows make 24 hours; Tuesday 9:00 gives z = 1.1, no alert; Monday 8:15 after an empty Sunday gives z = 9.3, alert; the weekly count stays between 38 and 49](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab05_rolling_z.gif)

**Expected output** *(computed from the data file when this notebook was built)*:

- **Line 1:** z-score alerts: about 393 to 416 (it depends how the code handles the first day, before a full 24 hours exists).
- **Line 2:** fixed alerts: 1,873.
- **The chart:** the CO2 line with far fewer flags than Detector 1.

**Compare your outcome.** Read the line *z-score alerts*, and find `rolling(` in Gemini's code:

| | z-score alerts | in Gemini's code | what it means |
|---|---|---|---|
| **A** | about 393 to 416 | `rolling(288` or `"24h"` | **Right:** 24 hours, looking back only. |
| **B** | about 358 | `rolling(24)` | **Wrong:** 24 rows is only 2 hours. |
| **C** | about 6 | `center=True` | **Wrong:** the window uses readings from the future. |

**If you got A:** read *What you should see* below, then go on to Task 3.

**If you got B or C:** in the same chat, type one sentence with your numbers, for example:
- for **B**: *alert_z has 358 alerts and the code uses rolling(24). The rows are 5
  minutes apart, so 24 hours is 288 rows. Use the last 288 rows and keep everything else.*
- for **C**: *alert_z has only 6 alerts and the code uses center=True, so it uses
  readings after the one it judges. Use a window that ends at each reading and keep
  everything else.*

Click **Accept and run**, then run the check cell again. Give the numbers, not just
*it didn't work*: with the numbers, Gemini knows what to fix; without them, it guesses.

**What you should see.** Far fewer flags: about 416 against 1,873. Every full week has
between 38 and 49, so the drift no longer makes the count climb: the window's average
drifts up with the sensor. But 382 of the 416 flags fall on Mondays, almost all
between 8:00 and 12:00. The 24 hours before a Monday morning are an empty Sunday, flat
at about 420 ppm, so the first people arriving look extreme.

## Task 3 — Build detector 3: compare with the same hour (20 minutes) · `PROMPT` `GENERATE` `VERIFY` `REFINE`

*Topic 5, section III: compare each reading with similar situations, such as the same hour of day.*

**Why a third detector?** Detector 2 raised 382 of its flags on Monday mornings,
only because it compared them with an empty Sunday. The same reading can be ordinary
at one hour and strange at another. In this file, CO2 at 11:00 averages 852 ppm; at
19:00, when the room is usually empty, it averages 511 ppm.

Task 3 has two parts: **3A**, build the detector; **3B**, see which alarms each
detector drops.

### 3A · Build the contextual detector

Compare each reading only with readings from the **same hour of the day**, on all
the other days: its z-score uses that hour's average and spread. A reading that is
odd for its hour, though not odd overall, is a **contextual anomaly**. You want
`alert_ctx`.

**Step 1.** `PROMPT` **Ask Gemini.** Open Gemini (the **spark icon** at the bottom centre of the Colab window), start a new chat (click **⋮** at the top right of the Gemini panel, then **New chat**, then **Yes**), paste this and press Enter. To copy it, drag
across the quoted text below from its first word to its last, then press Ctrl+C
(Cmd+C on a Mac):

> **Rules:** Do not drop any rows. Explain in plain words what the code does.
>
> **Task:** `df` has a `timestamp` column and a `co2_ppm` column. Group the readings
> by hour of the day (0 to 23). Within each hour, compute the mean and standard
> deviation of co2_ppm, then the z-score of each reading against its own hour. Make
> `alert_ctx`, True where the absolute z-score is above 3.

**Step 2.** `GENERATE` **Accept its code.** Gemini explains its plan in the chat and puts its code
into the notebook as a **green cell**. Click **Accept and run**: the button sits at
the bottom of the chat panel, just above the box where you type. It does not matter
where in the notebook the green cell lands. Ignore the blue suggestion chips under
its answer. If the code appears only in the chat, click the copy icon on the code,
click once on this text, click **+ Code** at the top of the page, paste into the new
cell, and run it. If Gemini ever shows red and green lines inside the check cell
(first line `# TODO(you)`), click **Cancel**: check cells are never changed.
Colab scrolls to the green cell, even to the very top of the notebook, so after
**Accept and run** scroll back down to this task for Step 3.

**Step 3.** `VERIFY` **Run the check cell below.** You should see the fewest alerts of the three (under twenty) and a chart with a few flags.

**Step 4.** `REFINE` **Only if a check fails, a number looks wrong, or Gemini's green cell
shows red error text:** in the same chat, type one
sentence saying what you saw and what you expected, for example *alert_ctx has 0 alerts; group by the hour of the day of each timestamp (0 to 23), not by the timestamp itself.* Press Enter,
click **Accept and run** (Gemini may add a new green cell or change its own earlier
one; either is fine), and run the check cell below again. If everything
passed, there is nothing to fix: move on.

In [ ]:
# TODO(you): Accept and run Gemini's green cell first (wherever it landed) (it makes `alert_ctx`), then run this cell.
alert_ctx = keep("alert_ctx")

if filled(alert_ctx):
    print(f"contextual alerts: {alert_ctx.sum():,}")
    show.flags(df, alert_ctx, col="co2_ppm", title="Detector 3: z-score within hour-of-day groups")

**What the cell above just did**, on two real readings (it plays slowly, about a minute):
![Contextual z-score: each hour's average and spread; Monday 8:15 at 585 ppm is ordinary for 8:00 (z = -0.3); 25 June 19:00 at 741 ppm is rare for 19:00 (z = 3.3)](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab05_same_hour.gif)

**Expected output** *(computed from the data file when this notebook was built)*:

- **Line 1:** contextual alerts: 7.
- **The chart:** the CO2 line with a few flags, all in the last week of June.
- **If it says 0:** the groups were made by the timestamp, not by the hour of day. Use the fix sentence in Step 4.

**What you should see.** The fewest flags of the three: 7. Detector 2's Monday-morning
flags are gone. At 8:15 on Monday 18 May, 585 ppm had a rolling z-score of 9.3; against
every 8 o'clock reading (average 640, spread 165) its z-score is −0.3, which is ordinary.
The 7 that remain are all in the evening or at night, on 25, 28 and 29 June, with the
room empty. By then the drift has lifted night-time CO2 above what is usual for that
hour.

### 3B · See which alarms each detector drops · `VERIFY`

Counts alone do not show *which* alerts each detector drops. **Run the cell below.**
It plots one week of CO2, Monday 15 to Sunday 21 June, with a row of tick marks per
detector. The legend gives each row's count.

**What you should see:** two rows of ticks above the line, and a legend with three
counts:

- **fixed:** 383;
- **rolling z:** about 44, and 42 of them on the Monday morning;
- **contextual z:** 0, so its row has no ticks.

In [ ]:
if filled(alert_fixed, alert_z, alert_ctx):
    week = (df.timestamp >= "2026-06-15") & (df.timestamp < "2026-06-22")
    fig, ax = plt.subplots(figsize=(11, 3.5))
    ax.plot(df.timestamp[week], df.co2_ppm[week], color="0.6", lw=0.8)
    for name, mask, y in [("fixed", alert_fixed, 1280), ("rolling z", alert_z, 1240),
                          ("contextual z", alert_ctx, 1200)]:
        m = week & mask
        ax.plot(df.timestamp[m], [y] * int(m.sum()), "|", ms=8, label=f"{name} ({int(m.sum())})")
    ax.axhline(1000, ls="--", color="0.4", lw=0.7)
    ax.set_ylabel("co2_ppm"); ax.legend(loc="lower left"); plt.show()

## Task 4 — Compare the three detectors (15 minutes) · `VERIFY`

*Topic 5, section V: every detector makes two kinds of mistake.* A **false alarm** is
an alert when nothing was wrong. A **missed detection** is no alert when something was.

1. `VERIFY` **Run the cell below.** It prints, for each detector, how many alerts it fired
   and how many of those fired while the room was empty. Under each detector, a second
   line compares the last 3 weeks with the first weeks.
2. **Copy those numbers** into the table on the sheet (or double-click the table
   under the cell and type them in).
3. `VERIFY` **Fill the last two columns** with yes or no.
   - *Fooled by the drift?* Read the detector's second line (it starts *last 3 weeks
     against first weeks*). Yes if it says 1.5 times or more; otherwise no. If it says
     *no alerts in the first weeks*, write *no*: the detector barely fires at all.
   - *Catches the stuck sensor?* No for all three: they watch CO2, and the stuck
     sensor is in `temp_c`.

In [ ]:
if filled(alert_fixed, alert_z, alert_ctx):
    empty = df.occupancy == 0
    late = df.timestamp >= df.timestamp.max() - pd.Timedelta(days=21)
    for name, mask in [("Fixed threshold", alert_fixed), ("Rolling z-score", alert_z), ("Contextual z-score", alert_ctx)]:
        early_rate = mask[~late].mean() * 100; late_rate = mask[late].mean() * 100
        print(f"{name:20s} alerts fired: {int(mask.sum()):5,}   fired when room empty: {int((mask & empty).sum()):4,}   "
              f"% of readings flagged: first weeks {early_rate:4.1f}, last 3 weeks {late_rate:4.1f}")
        print(f"{'':20s} last 3 weeks against first weeks: " + (f"{late_rate / early_rate:.1f} times" if early_rate else "no alerts in the first weeks"))

| Detector | Alerts fired | Fired when room empty | Fooled by the drift? | Catches the stuck sensor? |
|---|---|---|---|---|
| Fixed threshold | | | | |
| Rolling z-score | | | | |
| Contextual z-score | | | | |

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

- **Fixed threshold:** 1,873 alerts, 20 while the room was empty. Fooled by the drift:
  **yes**. Its rate goes from 7.0% of readings in the first weeks to 18.1% in the last
  three: 2.6 times. Most of those extra alerts are **false
  alarms**: the sensor drifted, the room did not change.
- **Rolling z-score:** about 393 to 416 alerts (it depends how Gemini started the
  24-hour window), 15 while empty. Fooled: **no**. Its rate goes from 2.6% to 2.1%,
  0.8 times (2.4% to 2.1% for 393, 0.9 times).
- **Contextual z-score:** 7 alerts, all 7 while empty. Fooled: **no** (no alerts in
  the first weeks, 0.1% in the last three, so it barely fires).
- **Stuck sensor:** **no** for all three: they watch CO2. Fourteen hours of a stuck
  `temp_c` with no alert is a **missed detection**.

The common slip is to write yes for the contextual detector because 7 of 7 fired
while empty. That is not the drift question. Those 7 are evening and night readings
in the last week of June that sit high for their hour.

**The comparison, step by step** (it plays slowly, about a minute):
![The three detectors compared: the fixed threshold's rate rises from 7.0% to 18.1% with the drift (false alarms); the stuck sensor is missed by all three; moving the z-score cut-off from 2 to 4 cuts alerts from 789 to 261](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab05_two_errors.gif)
</details>

## Task 5 — Build a detector for the stuck sensor (15 minutes) · `PROMPT` `GENERATE` `VERIFY` `REFINE`

*Topic 5: different faults need different checks.*

**None of the three catches the stuck sensor.** The stuck sensor from Lab 4 is in
**`temp_c`**, not CO2: on 14 May it reports 23.87 °C for fourteen hours, 9:00 to
22:55. That is not an outlier: at noon its rolling 24-hour z-score (the Detector 2
method, on `temp_c`) is only −1.0, because 23.87 °C is an ordinary temperature. What it loses is its wobble. So the number to watch is the
spread (standard deviation) over the last hour: on a stuck sensor it is exactly zero.
You want `stuck`.

**Step 1.** `PROMPT` **Ask Gemini.** Open Gemini (the **spark icon** at the bottom centre of the Colab window), start a new chat (click **⋮** at the top right of the Gemini panel, then **New chat**, then **Yes**), paste this and press Enter. To copy it, drag
across the quoted text below from its first word to its last, then press Ctrl+C
(Cmd+C on a Mac):

> **Rules:** Do not drop any rows. Explain in plain words what the code does.
>
> **Task:** `df` has a `temp_c` column with one row every five minutes. Compute the
> rolling standard deviation of temp_c over a 1-hour window (12 readings), and make
> `stuck`, True where that rolling standard deviation is exactly zero.

**Step 2.** `GENERATE` **Accept its code.** Gemini explains its plan in the chat and puts its code
into the notebook as a **green cell**. Click **Accept and run**: the button sits at
the bottom of the chat panel, just above the box where you type. It does not matter
where in the notebook the green cell lands. Ignore the blue suggestion chips under
its answer. If the code appears only in the chat, click the copy icon on the code,
click once on this text, click **+ Code** at the top of the page, paste into the new
cell, and run it. If Gemini ever shows red and green lines inside the check cell
(first line `# TODO(you)`), click **Cancel**: check cells are never changed.
Colab scrolls to the green cell, even to the very top of the notebook, so after
**Accept and run** scroll back down to this task for Step 3.

**Step 3.** `VERIFY` **Run the check cell below.** You should see about 157 readings flagged, all on 14 May, and `[PASS] flagged_at_least`.

**Step 4.** `REFINE` **Only if a check fails, a number looks wrong, or Gemini's green cell
shows red error text:** in the same chat, type one
sentence saying what you saw and what you expected, for example *stuck flags 281 readings, and 113 of them are not on 14 May; it flags every reading equal to the one before. Flag only where the rolling standard deviation of the last 12 readings is exactly 0.* Press Enter,
click **Accept and run** (Gemini may add a new green cell or change its own earlier
one; either is fine), and run the check cell below again. If everything
passed, there is nothing to fix: move on.

In [ ]:
# TODO(you): Accept and run Gemini's green cell first (wherever it landed) (it makes `stuck`), then run this cell.
stuck = keep("stuck")

if filled(stuck):
    print(f"stuck-sensor readings flagged: {int(stuck.sum())}")
    show.flags(df, stuck, col="temp_c", title="The stuck-sensor detector on temperature")

# ---- verification contract — do not edit ----
check.flagged_at_least(stuck, minimum=100, what="stuck readings")
check.report()

**What the cell above just did**, on 14 May (it plays slowly, about a minute):
![A stuck sensor: temp_c reads 23.87 for 168 readings; its z-score is only -1.0, but the spread of the last hour drops from about 0.45 to exactly 0, so 157 readings are flagged](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab05_stuck_flat.gif)

**Expected output** *(computed from the data file when this notebook was built)*:

- **Line 1:** stuck-sensor readings flagged: 157.
- **The chart:** temperature with one solid block of flags on 14 May.
- **The report:** `[PASS] flagged_at_least(stuck readings)`, then *All checks passed*.

**What you should see.** One solid block of 157 flags on 14 May, about 13 hours long,
on a line that otherwise wobbles normally. The stuck run is 168 readings (14 hours),
but the first 11 are not flagged: the 1-hour window needs 12 equal readings before its
spread is 0, and 168 − 11 = 157. If some flags are not on 14 May, the code flags every
reading equal to the one before (281 flags): use the fix sentence in Step 4.

---
## Predict: What will the busiest hour's taxi count be? (15 minutes)

*The Prediction League: one question per lab, settled by reality. Individual work: you enter on your own, under your own name.*

**The question:** The largest hourly taxi-availability count during the target week. The target is the week of Monday 2026-11-30.

**How it works:** you enter a short function, `predict(history)`, not a number. When
entries close, the league runs your function once on the latest data. When the target
is over, the real value is compared with yours. League page (with the closing time):
https://stanleysp-eng.github.io/EP0418-labs/predict.html?lab=5

**Step 1.** **Run the cell below.** It prints the number from the simple rule,
*the largest hourly count in the last seven days*, on the lab's own file. The league works the simple rule out again
from the latest data when entries close, so its number will differ.

In [ ]:
fetch("sg_taxi_hourly.csv")
taxi = pd.read_csv("sg_taxi_hourly.csv", parse_dates=["timestamp"])

import pandas as pd

def predict(history):
    # the baseline: the busiest hour of the last seven days
    last_week = history[history["timestamp"] > history["timestamp"].max() - pd.Timedelta(days=7)]
    print("hours in the last week:", len(last_week))
    return float(last_week["taxi_count"].max())

print("The baseline predicts", round(predict(taxi), 1), "taxis")

**What you should see:** *The baseline predicts 3605.0 taxis*.

**Step 2.** **Ask Gemini to write your predict.** Start a new
chat (**⋮** › **New chat** › **Yes**), paste the prompt and press Enter.

> **Rules:** Put it in a new cell and do not change the cells above. Write only the function and any import lines it needs. Keep the name *predict* and its one input, *history*, a pandas table with the columns timestamp (one reading per hour, Singapore time) and taxi_count, oldest first. Return one number with float(...). **Task:** Write a predict(history) that estimates this: the largest hourly taxi-availability count during the target week. Do it in a way you think beats the simple rule (the largest hourly count in the last seven days). Explain the idea in one sentence.

Click **Accept and run** on Gemini's green cell. It prints nothing; it only defines the
function. If Gemini shows red and green lines inside the Step 1 cell, click **Cancel** and type *put it in a new cell*.

**Step 3.** **Run the cell below.**

In [ ]:
# TODO(you): run Gemini's green cell first (Step 2)
show_predict(taxi, "taxis")

**What you should see:** *Your predict says* and a number, usually between 3,000 and 4,500, then your
code between two dashed lines. If it says *Gemini's predict has not run yet*, do what that message says.
If you see a red error, paste the last red line into Gemini with *fix this*, click
**Accept and run**, and run this cell again.

<details><summary><b>Using ChatGPT, Claude or another AI instead of Gemini?</b> Open this for a prompt to copy</summary>

That AI cannot see this notebook, so it needs a longer prompt that spells out every
rule the league checks. Use it in place of Steps 2 and 3:

1. **Copy the prompt** in the grey box below.
2. **Paste it** into the other AI's chat box. Do not send it yet.
3. **Write your idea.** In the line *My idea: WRITE YOUR IDEA HERE*, replace *WRITE YOUR IDEA HERE* with your idea, in plain words.
   Plain text is fine: no bold or italics needed. If you have no idea yet, delete the whole line: the AI then chooses one.
4. **Optional:** to let the AI look at the numbers first, download the data,
   https://stanleysp-eng.github.io/EP0418-labs/data/taxi_hourly.csv (the same file your
   function receives, updated every night), and attach it to the chat.
5. **Send it.** Copy the code the AI gives back, then do **Step 4** below: paste that
   code into the league page's **Code** box, in place of the code from Step 3.

```text
I am entering a prediction contest. Write one Python function for it.

The question: The largest hourly taxi-availability count during the target week. The target is the week of Monday 2026-11-30 to Sunday 2026-12-06.
The simple rule to beat: the largest hourly count in the last seven days.
My idea: WRITE YOUR IDEA HERE

The contest runs the function automatically, so follow these rules exactly:
1. Name it predict, with exactly one input called history:  def predict(history):
2. history is a pandas DataFrame that the contest passes in, oldest row first. Its column names are exactly: timestamp, taxi_count. What they hold: timestamp (one reading per hour, Singapore time) and taxi_count. The timestamp column is timezone-aware (Asia/Singapore): compare it only with timezone-aware times, for example pd.Timestamp("2026-10-26", tz="Asia/Singapore").
3. Do not read any file and do not ask for input. All the data is already in history. If I attach a data file, look at it only to choose the idea; the function must still take everything from history.
4. Return one number with return float(...). Not a list, a table, text, None or NaN.
5. Use only pandas, numpy and the Python standard library, and finish in under 30 seconds.
6. Give only the import lines and the function: no example data, no test code, nothing that calls predict.

After the code, explain the idea in one sentence.
```
</details>

**Step 4. Submit on the league page.**
1. **Copy your code** from the Step 3 output: everything between the two dashed lines.
2. **Open** https://stanleysp-eng.github.io/EP0418-labs/predict.html?lab=5 in a new tab and type your name as on the class list (the same name every week).
3. **Paste** it into the **Code** box: click in it, Ctrl+A, Ctrl+V (Cmd on a Mac).
4. **Click Run tests** (about 15 seconds the first time). **What you should see:** *All 4 cases ran* and four tabs: three past cases and *Your entry*. On *Your entry*, Actual says *not known yet*, and the number differs from Colab's because the page uses newer data: both are normal. A red *missed by* only means the simple rule did better on that case. A **red heading** instead of *All 4 cases ran* is an error: copy the red text under the case marked red (or under the heading), paste it into the same Gemini chat with *fix this*, click **Accept and run**, run Step 3 again, then paste and **Run tests** again.
5. **Click Submit.** *Submitted at ...* confirms it. Submit again any time before the close; the latest one counts.

**Write it down** (on the sheet, or double-click here): your function's number (Step 3),
the baseline (Step 1), and the one-sentence idea Gemini gave. After the target, the
real answer is on the league page.

**My number / the baseline / the idea:**

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

There is no single right answer here. A good entry gives all three parts: your
function's number, the baseline (3605.0 taxis), and an idea that uses the data. The busiest hour of each week in the file was between 3,434 and 4,159, so it moves by several hundred taxis from week to week. Ideas that use this: the average of the recent weekly maxima, or the largest count at the busiest hour of day over the last few weeks.
The common weak answer is "Gemini wrote it". It says who wrote the code, not what the idea is. The past-case
tabs on the league page show whether the idea would have beaten the simple rule
lately; three cases are too few to prove it.
</details>

---
## Read before you run

An assistant was asked for a rolling z-score of CO2 over a 24-hour window. It wrote the code below.

The code is switched off: every line starts with `#`, so it cannot run yet.

1. **Read it. Do not run it.** Answer questions 1 and 2 in the box under the code
   cell, or on the sheet.
2. **Ask Gemini to explain it.** Click in the code cell, press Ctrl+A then Ctrl+C
   (Cmd on a Mac). In Gemini, start a new chat (**⋮** › **New chat** › **Yes**),
   paste the code into the chat box, add *Explain this code in plain words.* and
   press Enter. If Gemini offers a green cell or red and green lines, click
   **Cancel**: this step only explains. Check whether the explanation points out the problem.
   Often it does not.
3. **Then run it.** Delete the `# ` at the start of each line below the first line,
   run the cell, and see whether you were right.

**Question 1.** The comment says 24 hours. Each row is five minutes apart. How long is the window really?

**Question 2.** Write down the number that should go inside `rolling( )` for a 24-hour window on this data (readings every 5 minutes).

<details><summary>Hint 1: where to look</summary>

`rolling(24)` counts rows, not hours. How many rows make one hour at five-minute spacing?

</details>
<details><summary>Hint 2: closer to the answer</summary>

Twelve rows an hour, so 24 rows is two hours. A 24-hour window is 288 rows, or `rolling("24h")` on a time index.

</details>

An explanation that reads well is not evidence the code is right. 

**Optional, to fix it:** in the same chat, type *Fix the problem in this code that
Hint 2 describes. Say in one sentence what you changed.* If that one change matches
Hint 2, click **Accept and run** (accepting is right here: you asked for the fix). If
it changed other things too, click **Cancel** and type *fix only that, nothing else*.
You should see: the window is now 288 rows, 24 hours. The alert count moves only a little (about 358 with the 2-hour window, about 400 with 24 hours): a wrong window can still give a believable count, so check the window size itself, not just the count. (The hours line may still say 2.0 if Gemini did not change that line.)

In [ ]:
# ---- Every line below this one is the code. Do steps 1 and 2 first. For step 3, delete the "# " at the start of each line below, then run the cell. ----
# r = df.co2_ppm.rolling(24)          # 24-hour window
# their_z = (df.co2_ppm - r.mean()) / r.std()
# print("this window covers", 24 * 5 / 60, "hours of readings")
# print("alerts with this window:", int((their_z.abs() > 3).sum()), " | yours from Detector 2:", int(alert_z.sum()) if alert_z is not None else "not run")

**Answer to question 1:**

**Answer to question 2:**

**Did Gemini's explanation mention the fault? (yes / no, and the sentence):**

**Was I right, once I ran it?**

---
## Quiz yourself before you leave (5 minutes)

The checks tested the code. This quiz tests what you understood. In Gemini, start a new chat
(**⋮** › **New chat** › **Yes**), paste this into the chat box and press Enter:

> **Rules:** Ask one short question at a time, in plain words. Take the examples from the code and outputs in this notebook. Wait for my answer. Then say in one sentence whether I am right, and why. Do not give the next question until I answer.
>
> **Task:** Quiz me on this lab: five questions on these ideas: what a z-score flags and what it misses; why a contextual detector raises fewer false alarms; why a stuck sensor produces no outliers; the difference between a false alarm and a missed detection; why the choice between them belongs to the person who pays for the mistake. After the fifth, give my score out of 5 and the one idea I should revise, in one line.

Gemini asks one question at a time in the chat panel; answer in the chat box, in
your own words, without looking back. Ignore the suggested replies Gemini shows. If you
use ChatGPT, Claude or Copilot instead, first delete the sentence *Take the examples from
the code and outputs in this notebook.*: they cannot see your notebook.

Then write your score and the idea to revise below. Your tutor asks for them.

**Quiz score:** ___ / 5

**The idea I should revise, in my own words:**

*(Your answer box: double-click this cell. Type your answer here.)*

---
## Before you leave

Double-click the box cell below and type there, or write on the last page of the Lab 5
sheet:

- **Mistake caught:** one thing Gemini got wrong today and how you saw it. If it got
  nothing wrong, use the read-before-you-run code instead.
- **Decision:** which detector you would actually deploy for a classroom CO2 alarm,
  with its cut-off, and the mistake it will still make: false alarms, missed
  detections, or both. Which is worse there: a false alarm, or a missed detection?

**Optional, the Trading League:** a z-score of recent daily price moves is a
volatility filter: hold the share, but go to cash when the z-score is large. If you
are in the League, open your League notebook and try it as your Task 4 rule.

**Mistake caught:** *(three short lines: 1. what I expected, 2. what was there instead, 3. how I found it. If Gemini made no mistake today, use the read-before-you-run code above instead.)*

**Decision:** *(what I decided · why · the evidence)*

*(Your answer box: double-click this cell. Type your answer here.)*